# 🎨 TwitCanva no Google Colab

Este notebook baixa todos os arquivos do repositório **TwitCanva-Video-Workflow**, instala as dependências e inicia o aplicativo (backend + frontend).

## Como usar
1. Execute as células em ordem (Shift+Enter).
2. A Célula 1 baixa o projeto de 2 formas:
   - **Opção A (recomendada)** — clona direto do GitHub (repositório público).
   - **Opção B** — se preferir, faça upload manual do `TwitCanva-Video-Workflow.zip` (basta rodar a célula de upload); o notebook detecta e descompacta.
3. Na célula de `.env`, cole suas API keys (obrigatório para gerar imagens/vídeos).
4. A última célula inicia o servidor e abre o TwitCanva em uma nova janela do navegador.

> 💡 O Colab mantém os arquivos apenas enquanto a sessão estiver ativa. Salve seus fluxos/gerações em `library/` ou faça download antes de encerrar.

## ✅ Célula 1 — Obter todos os arquivos do projeto

In [ ]:
# Opção A: clonar do GitHub
!rm -rf /content/TwitCanva-Video-Workflow
!git clone --depth 1 https://github.com/anonyby777-lgtm/TwitCanva-Video-Workflow.git /content/TwitCanva-Video-Workflow
!cd /content/TwitCanva-Video-Workflow && git checkout main
print('Projeto baixado com sucesso!')
!ls -la /content/TwitCanva-Video-Workflow

In [ ]:
# Opção B (opcional): se preferir, envie o ZIP manualmente
# Execute esta célula SOMENTE se não quiser usar o clone do GitHub.
# from google.colab import files
# uploaded = files.upload()  # selecione TwitCanva-Video-Workflow.zip
# !mkdir -p /content && !unzip -q -o TwitCanva-Video-Workflow.zip -d /content
# print('ZIP extraído em /content/TwitCanva-Video-Workflow')
print('Pulando (use a Opção A acima para baixar do GitHub).')

## ✅ Célula 2 — Verificar/instalar Node.js (necessário para o app)

In [ ]:
%%bash
NODE_MAJOR=$(node -v 2>/dev/null | sed -E 's/v([0-9]+).*/\1/')
if [ -z "$NODE_MAJOR" ] || [ "$NODE_MAJOR" -lt 18 ]; then
  echo 'Instalando Node.js 20...'
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  apt-get install -y -qq nodejs
else
  echo "Node.js $NODE_MAJOR já instalado."
fi
node -v && npm -v

## ✅ Célula 3 — Instalar dependências do projeto frontend/backend (Node)

In [ ]:
%cd /content/TwitCanva-Video-Workflow
!npm ci  # instala todas as dependências do package-lock.json

## ✅ Célula 4 — Configurar chaves de API (`.env`)

Sem essas chaves o app abre, mas a geração de imagens/vídeos não funciona. As chaves ficam salvas em `.env` na raiz do projeto.

- `GEMINI_API_KEY` — Google Gemini / GPT Image (via Google)
- `KLING_ACCESS_KEY` / `KLING_SECRET_KEY` — Kling AI (vídeos)
- `HAILUO_API_KEY` — Hailuo AI / MiniMax (vídeos)
- `OPENAI_API_KEY` — OpenAI (GPT Image)
- `FAL_API_KEY` — Fal.ai (Veo, Seedance, movimento)
- `TWITTER_API_KEY` / `TWITTER_API_SECRET` ... — opcional, apenas para postar no X
- `TIKTOK_CLIENT_KEY` / `TIKTOK_CLIENT_SECRET` ... — opcional, apenas para TikTok

> Todas são opcionais, mas quanto mais chaves você preencher, mais modelos ficam disponíveis.

In [ ]:
import os
from getpass import getpass

print('Deixe em branco os campos que você não tem.')
keys = {
    'GEMINI_API_KEY': 'Google Gemini',
    'KLING_ACCESS_KEY': 'Kling Access Key',
    'KLING_SECRET_KEY': 'Kling Secret Key',
    'HAILUO_API_KEY': 'Hailuo / MiniMax',
    'OPENAI_API_KEY': 'OpenAI',
    'FAL_API_KEY': 'Fal.ai',
    'TWITTER_CLIENT_ID': 'X/Twitter Client ID (opcional)',
    'TWITTER_CLIENT_SECRET': 'X/Twitter Client Secret (opcional)',
    'TWITTER_API_KEY': 'X/Twitter API Key (opcional)',
    'TWITTER_API_SECRET': 'X/Twitter API Secret (opcional)',
    'TWITTER_ACCESS_TOKEN': 'X/Twitter Access Token (opcional)',
    'TWITTER_ACCESS_TOKEN_SECRET': 'X/Twitter Access Token Secret (opcional)',
    'TIKTOK_CLIENT_KEY': 'TikTok Client Key (opcional)',
    'TIKTOK_CLIENT_SECRET': 'TikTok Client Secret (opcional)',
}

env_path = '/content/TwitCanva-Video-Workflow/.env'
existing = {}
if os.path.exists(env_path):
    for line in open(env_path):
        line = line.strip()
        if line and '=' in line and not line.startswith('#'):
            k, v = line.split('=', 1)
            existing[k.strip()] = v.strip()

with open(env_path, 'w') as f:
    f.write('# TwitCanva - chaves de API\n')
    for k, label in keys.items():
        default = existing.get(k, '')
        val = getpass(f'{label} [{k}]: ') or default
        f.write(f'{k}={val}\n')

print('\n.env salvo em', env_path)

## ✅ Célula 5 — Compilar o frontend (build de produção)

In [ ]:
%cd /content/TwitCanva-Video-Workflow
!npm run build
print('Build concluído! (pasta dist/)')

## ✅ Célula 6 — Iniciar o TwitCanva 🚀

Esta célula sobe o backend (porta 3001) e abre o app em uma janela do navegador.

In [ ]:
import os, subprocess, time, signal

os.chdir('/content/TwitCanva-Video-Workflow')
env = dict(os.environ)
env['NODE_ENV'] = 'production'
env.pop('PORT', None)

with open('/content/twitcanva-server.log', 'w') as log:
    proc = subprocess.Popen(
        ['node', 'server/index.js'],
        stdout=log, stderr=subprocess.STDOUT,
        env=env, start_new_session=True
    )

# Espera o servidor subir
for _ in range(30):
    if proc.poll() is not None:
        break
    line = open('/content/twitcanva-server.log').read()
    if 'Backend server running' in line:
        break
    time.sleep(1)

print('Log do servidor:')
!tail -15 /content/twitcanva-server.log

# Abre o app em uma aba do navegador (porta 3001)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(3001, height=900)
print('TwitCanva iniciado! Se a janela não abrir, use a URL acima.')

# Para reiniciar depois: clique em Célula 6 novamente (o processo antigo é encerrado ao final da sessão).

## ✅ (Opcional) Célula 7 — Modelos locais (Python)

Só é necessária se você quiser usar Stable Diffusion / Qwen locais. O Colab já vem com PyTorch; esta célula instala o restante de `requirements.txt`.

In [ ]:
%cd /content/TwitCanva-Video-Workflow
!pip install -q -r requirements.txt
print('Dependências Python instaladas.')

## 💡 Dicas

- **Reiniciar o app:** execute a Célula 6 novamente.
- **Salvar seu trabalho:** os arquivos gerados ficam em `library/` dentro do projeto. Baixe com `files.download()` ou monte o Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) e copie a pasta `library/`.
- **Instagram/TikTok/X:** configure as chaves opcionais na Célula 4 e ajuste os callbacks conforme `docs/`.
- **Problemas de memória/GPU:** como o frontend não usa GPU, pode rodar em qualquer runtime (`CPU` ou `GPU`). Modelos locais exigem `GPU (T4)`.
- **Portais:** o backend escuta em `0.0.0.0:3001` e o Colab serve a janela pelo proxy nativo.